
# TP 4 — Losses & initialisation

**Objectifs de la séance.**
- Passer de la régression à la classification : choisir la bonne fonction de perte et la bonne activation de sortie (MSE, BCE, CrossEntropy).
- Comparer classification binaire et multiclasse.
- Comparer différentes stratégies d'initialisation des poids (naïve vs Xavier/He) et leur effet sur la vitesse de convergence.
- Suivre une métrique (accuracy) en plus de la loss, avec le `Trainer`.

À partir de cette séance, on utilise systématiquement `from training_toolbox import Trainer, accuracy`.

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Subset
import matplotlib.pyplot as plt

from training_toolbox import Trainer, accuracy

torch.manual_seed(0)


## Partie 1 — Régression : California housing

On reprend un problème de régression tabulaire, cette fois avec plusieurs variables d'entrée : le jeu de données *California housing* (prix médian de l'immobilier par quartier, à partir de variables comme le revenu médian, l'âge des logements, etc.). Le code ci-dessous charge et prétraite les données (normalisation min-max des entrées et de la cible, découpage train/validation) — pas besoin d'y toucher.

In [ ]:

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

data = fetch_california_housing()
X, y = data.data, data.target.reshape(-1, 1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

scaler_x = MinMaxScaler().fit(X_train)
X_train, X_val = scaler_x.transform(X_train), scaler_x.transform(X_val)

scaler_y = MinMaxScaler().fit(y_train)
y_train, y_val = scaler_y.transform(y_train), scaler_y.transform(y_val)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

reg_train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
reg_val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)

n_features = X_train_t.shape[1]
print("Nombre de variables d'entrée :", n_features)


**Question 1.1.** Définissez un MLP `RegressionMLP` (une ou deux couches cachées, activation ReLU) avec une **sortie unique et une activation de sortie identité** (pas d'activation après la dernière couche : la sortie doit pouvoir prendre n'importe quelle valeur réelle, positive ou négative). Entraînez-le avec le `Trainer`, `nn.MSELoss()` et `torch.optim.Adam`, pendant 15 époques sur `reg_train_loader`/`reg_val_loader`.

In [ ]:

class RegressionMLP(nn.Module):
    def __init__(self, n_features, hidden_size=64):
        super().__init__()
        # TODO : définir les couches (au moins une couche cachée + ReLU, puis une sortie unique
        #        SANS activation)

    def forward(self, x):
        # TODO : enchaîner les couches et renvoyer la sortie
        pass


# TODO : instancier le modèle, l'optimizer, le Trainer (nn.MSELoss()), et appeler .fit(...)


## Partie 2 — Classification : MNIST

On charge le jeu de données MNIST (chiffres manuscrits, 28x28 pixels) via `torchvision`. Les images sont aplaties en vecteurs de 784 valeurs (dans $[0, 1]$, `ToTensor` s'en charge) pour être utilisées par un MLP. Le code ci-dessous prépare deux versions : `mnist_train/val_loader` (les 10 classes) et `binary_train/val_loader` (seulement les chiffres 0 et 1, pour la classification binaire).

In [ ]:

from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

mnist_train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
mnist_test_full = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

mnist_train_loader = DataLoader(mnist_train_full, batch_size=128, shuffle=True)
mnist_val_loader = DataLoader(mnist_test_full, batch_size=128, shuffle=False)

# Sous-ensemble binaire : uniquement les chiffres 0 et 1
def binary_subset(dataset):
    mask = (dataset.targets == 0) | (dataset.targets == 1)
    indices = mask.nonzero(as_tuple=True)[0]
    return Subset(dataset, indices)

binary_train_loader = DataLoader(binary_subset(mnist_train_full), batch_size=128, shuffle=True)
binary_val_loader = DataLoader(binary_subset(mnist_test_full), batch_size=128, shuffle=False)

print("Exemple d'entrée :", mnist_train_full[0][0].shape, "- label :", mnist_train_full[0][1])


### Classification binaire (0 vs 1)

Pour une classification **binaire**, une convention courante est d'avoir une seule sortie (un score, ou *logit*), et d'utiliser `nn.BCEWithLogitsLoss` (BCE = *binary cross-entropy*), qui applique une sigmoïde en interne de façon numériquement stable — pas besoin d'ajouter vous-même une `nn.Sigmoid()` en sortie du modèle.

La métrique `accuracy` de `training_toolbox` suppose une sortie multiclasse (elle fait un `argmax`) : pour le cas binaire à une seule sortie, on définit une petite fonction dédiée (fournie ci-dessous, pas besoin d'y toucher).

In [ ]:

def binary_accuracy(preds, y):
    # preds : logits de forme (batch, 1) ; y : labels 0/1 de forme (batch,) ou (batch, 1)
    pred_labels = (preds > 0).float()
    return (pred_labels == y.float().view(-1, 1)).float().mean()


def bce_loss(preds, y):
    return nn.functional.binary_cross_entropy_with_logits(preds, y.float().view(-1, 1))


**Question 2.1.** Définissez un MLP `BinaryMLP` (une couche cachée, activation ReLU, **une seule sortie, sans activation en sortie**). Entraînez-le avec le `Trainer`, `loss_fn=bce_loss`, `torch.optim.Adam` et `metrics={"acc": binary_accuracy}`, pendant 5 époques sur `binary_train_loader`/`binary_val_loader`.

In [ ]:

class BinaryMLP(nn.Module):
    def __init__(self, n_features=784, hidden_size=64):
        super().__init__()
        # TODO : une couche cachée + ReLU, puis une couche de sortie à 1 neurone (sans activation)

    def forward(self, x):
        # TODO
        pass


# TODO : instancier le modèle, l'optimizer, le Trainer (loss_fn=bce_loss, metrics={"acc": binary_accuracy}),
#        et appeler .fit(binary_train_loader, binary_val_loader, epochs=5)


### Classification multiclasse (10 chiffres)

Pour une classification **multiclasse**, la sortie a autant de neurones que de classes (10 ici), et on utilise `nn.CrossEntropyLoss`, qui applique un softmax en interne de façon numériquement stable — là encore, pas de `nn.Softmax()` à ajouter en sortie du modèle, et les labels restent des entiers de classe (pas besoin de les encoder en *one-hot*).

**Question 2.2.** Définissez un MLP `MulticlassMLP` (une ou deux couches cachées, activation ReLU, **10 sorties, sans activation en sortie**). Entraînez-le avec `nn.CrossEntropyLoss()`, `torch.optim.Adam` et `metrics={"acc": accuracy}` (celle de `training_toolbox`, adaptée au cas multiclasse), pendant 5 époques sur `mnist_train_loader`/`mnist_val_loader`.

In [ ]:

class MulticlassMLP(nn.Module):
    def __init__(self, n_features=784, hidden_size=128, n_classes=10):
        super().__init__()
        # TODO : une ou deux couches cachées + ReLU, puis une couche de sortie à n_classes neurones
        #        (sans activation)

    def forward(self, x):
        # TODO
        pass


# TODO : instancier le modèle, l'optimizer, le Trainer (nn.CrossEntropyLoss(), metrics={"acc": accuracy}),
#        et appeler .fit(mnist_train_loader, mnist_val_loader, epochs=5)


**Questions.**
- Pourquoi ne met-on jamais de `nn.Sigmoid()`/`nn.Softmax()` explicite juste avant `BCEWithLogitsLoss`/`CrossEntropyLoss` ?
- Que se passerait-il si on utilisait `nn.MSELoss()` à la place de `CrossEntropyLoss` pour la classification multiclasse (en encodant les labels en *one-hot*) ? Quel type de gradient cela donnerait-il pour des prédictions déjà « presque correctes » mais pas encore parfaites ?


_Votre réponse ici._


## Partie 3 — Initialisation des poids

Par défaut, `nn.Linear` initialise déjà ses poids de façon raisonnable (une variante de He/Xavier selon la version de PyTorch). Pour observer l'effet d'une **mauvaise** initialisation, on va la provoquer volontairement, puis la corriger.

**Question 3.1.** Écrivez une fonction `init_naive(module)` qui, si `module` est une `nn.Linear`, réinitialise `module.weight` avec `nn.init.normal_(module.weight, mean=0.0, std=5.0)` (un écart-type volontairement bien trop grand) et met `module.bias` à zéro (`nn.init.zeros_`). Écrivez de même `init_he(module)` qui utilise `nn.init.kaiming_uniform_(module.weight, nonlinearity="relu")` (adapté aux couches suivies d'une ReLU).

In [ ]:

def init_naive(module):
    if isinstance(module, nn.Linear):
        pass  # TODO : nn.init.normal_ sur module.weight (std=5.0) et nn.init.zeros_ sur module.bias


def init_he(module):
    if isinstance(module, nn.Linear):
        pass  # TODO : nn.init.kaiming_uniform_ sur module.weight (nonlinearity="relu")
              #        et nn.init.zeros_ sur module.bias


**Question 3.2.** Créez deux `MulticlassMLP` neufs, avec la même architecture (par exemple deux couches cachées de 128 neurones). Appliquez `model.apply(init_naive)` au premier et `model.apply(init_he)` au second. Entraînez les deux (mêmes hyperparamètres, mêmes 5 époques) et comparez les courbes de perte et d'accuracy sur les premières époques.

In [ ]:

# TODO : deux MulticlassMLP neufs, l'un avec model.apply(init_naive), l'autre avec model.apply(init_he),
#        entraînés dans les mêmes conditions (5 époques) ; comparer les courbes train_loss / train_acc


**Questions.** Quel effet observez-vous sur la vitesse de convergence en tout début d'entraînement ? Une initialisation trop « large » (grande variance) pose-t-elle un problème particulier avec des activations ReLU ? Et avec des activations sigmoïde/tanh (où le gradient s'annule pour de grandes valeurs d'entrée) ?


_Votre réponse ici._


## Bilan

Vous savez maintenant choisir la loss et l'activation de sortie adaptées à un problème (régression, classification binaire, classification multiclasse), et vous avez observé l'effet de l'initialisation des poids sur la convergence. La séance 5 s'attaque au problème inverse : un modèle qui apprend trop bien le jeu d'entraînement (sur-apprentissage), et comment le corriger.